# Employee Growth Score (EGS)
### A Cumulative, Additive Contribution Model for 

This model implements the **Employee Growth Score (EGS)** — an employee
evaluation model built for simplicity, transparency, and operational adoption.

It is a **growth score model** with:
- fewer dimensions (8 total)
- event / achievement based
- **additive** points 
- the score **only grows over time** as people contribute
- importance is encoded in **point values**, not hidden weights
- there is **no upper cap** like 100

**Design principle:** Importance lives in the *point price* of an action, not in
a complex weighting formula. A routine task is worth 2–3 points; a business-critical
achievement is worth 15–20.


## The Core Formula

The lifetime score is simply the sum of everything earned, minus serious deductions:

$$
EGS = \sum_{i=1}^{5} P_i + \sum_{j=6}^{8} P_j + R - D
$$

Where:
- $$P_1 \dots P_5$$ = the **5 universal pillars** (apply to everyone)
- $$P_6 \dots P_8$$ = the **3 role-specific pillars**
- $$R$$ = recognition / milestone bonus points
- $$D$$ = deductions for serious failures only

### Cumulative carry-forward over time

The score grows month over month:

$$
EGS_{current} = EGS_{previous} + EGS_{period}
$$

where the points earned in a single period are:

$$
EGS_{period} = \sum_{i=1}^{8} P_i + R - D
$$


## The Eight Pillars

### 5 Universal Pillars (everyone)

| # | Pillar |
|---|--------|
| 1 | Delivery & Quality | 
| 2 | Collaboration & Culture | 
| 3 | Ownership & Impact | 
| 4 | Learning & Growth | 
| 5 | Reliability & Work Discipline | 11, 12 |

### 3 Role-Specific Pillars (per role)

| # | Generic name | Meaning |
|---|--------------|---------|
| 6 | Role Output | Core measurable outcomes |
| 7 | Role Process Excellence | Doing the work the right way |
| 8 | Role Strategic Impact | Long-term / strategic value |

Roles supported: **Sales, Projects, HR, Accounts, Team Managers, Vendor Managers**.


## The Scoring Engine

I am modeling each employee as a **scorecard** object that:
1. Accumulates points by category during a period (a month).
2. Lets us add or deduct points with a category label.
3. **Closes the month** — rolling the monthly score into a lifetime cumulative total
   and returning a clean breakdown.

This keeps the running history intact while giving a fresh monthly view each cycle.


In [2]:
from dataclasses import dataclass, field
from typing import Dict, List
from datetime import datetime

@dataclass
class EmployeeScoreCard:
    name: str
    role: str
    cumulative_score: int = 0
    monthly_score: int = 0
    breakdown: Dict[str, int] = field(default_factory=dict)
    history: List[dict] = field(default_factory=list)

    def add_points(self, category: str, points: int) :
        if points <= 0:
            return
        self.breakdown[category] = self.breakdown.get(category, 0) + points
        self.monthly_score += points

    def deduct_points(self, category: str, points: int):
        if points <= 0:
            return
        self.breakdown[category] = self.breakdown.get(category, 0) - points
        self.monthly_score -= points

    def close_month(self, label: str | None = None):
        self.cumulative_score += self.monthly_score
        snapshot = {
            "label": label or datetime.now().strftime("%Y-%m"),
            "name": self.name,
            "role": self.role,
            "monthly_score": self.monthly_score,
            "cumulative_score": self.cumulative_score,
            "breakdown": dict(self.breakdown),
        }
        self.history.append(snapshot)
        self.monthly_score = 0
        self.breakdown = {}
        return snapshot 


## Universal Pillar 1 — Delivery & Quality

Measures whether work gets done properly and on time.

### Point rules

| Action | Points |
|---|---:|
| Completed a planned task | +2 |
| High-priority task completed on time | +4 |
| Major deliverable completed on time | +8 |
| Delivered with no rework required | +3 |
| Accepted in first review | +4 |
| Above-benchmark complexity completed | +5 |
| Early completion with acceptable quality | +2 |

### Deductions (major failures only)

| Issue | Points |
|---|---:|
| Deliverable required major rework | −4 |
| Missed critical deadline | −6 |
| Severe quality defect (business/client impact) | −8 |

### Formula

$$
P_1 = 2T + 4H + 8D + 3N + 4F + 5C + 2E - 4R_w - 6M - 8Q_f
$$


In [12]:
def score_delivery_quality(
    card,
    tasks_completed=0,
    high_priority_on_time=0,
    major_deliverables_on_time=0,
    no_rework=0,
    first_pass_accept=0,
    complex_work=0,
    early_completion=0,
    major_rework=0,
    critical_miss=0,
    severe_quality_failure=0,
):
    cat = "Delivery & Quality"
    card.add_points(cat, 2 * tasks_completed)
    card.add_points(cat, 4 * high_priority_on_time)
    card.add_points(cat, 8 * major_deliverables_on_time)
    card.add_points(cat, 3 * no_rework)
    card.add_points(cat, 4 * first_pass_accept)
    card.add_points(cat, 5 * complex_work)
    card.add_points(cat, 2 * early_completion)
    card.deduct_points(cat, 4 * major_rework)
    card.deduct_points(cat, 6 * critical_miss)
    card.deduct_points(cat, 8 * severe_quality_failure)


## Universal Pillar 2 — Collaboration & Culture

Measures teamwork and positive contribution to the working environment.

### Point rules

| Action | Points |
|---|---:|
| Supported a teammate meaningfully | +3 |
| Shared knowledge / documentation / training | +4 |
| Positive peer feedback instance | +3 |
| Helped resolve a team blocker | +5 |
| Participated in a non-mandatory culture initiative | +2 |
| Mentored a junior / new team member | +5 |

**Deductions:**

| Issue | Points |
|---|---:|
| Collaboration failurey | -4 |
| repeated communication lapse | -3 |
| confirmed negative behavior incident | -8 |

### Formula

$$
P_2 = 3S + 4K + 3P_f + 5B + 2C_u + 5M_t - 4C_f - 3L - 8N_g
$$


In [13]:
def score_collaboration_culture(
    card,
    teammate_support=0,
    knowledge_shares=0,
    positive_peer_feedback=0,
    blockers_resolved=0,
    culture_participation=0,
    mentoring=0,
    collab_failures=0,
    comm_lapses=0,
    negative_incidents=0,
):
    cat = "Collaboration & Culture"
    card.add_points(cat, 3 * teammate_support)
    card.add_points(cat, 4 * knowledge_shares)
    card.add_points(cat, 3 * positive_peer_feedback)
    card.add_points(cat, 5 * blockers_resolved)
    card.add_points(cat, 2 * culture_participation)
    card.add_points(cat, 5 * mentoring)
    card.deduct_points(cat, 4 * collab_failures)
    card.deduct_points(cat, 3 * comm_lapses)
    card.deduct_points(cat, 8 * negative_incidents)


## Universal Pillar 3 — Ownership & Impact

Measures initiative, problem-solving, responsibility, and business impact.

### Point rules

| Action | Points |
|---|---:|
| Identified & solved a problem independently | +5 |
| Proposed a useful improvement | +4 |
| Implemented an approved improvement | +8 |
| Took ownership beyond assigned scope | +5 |
| Proactively prevented a risk / escalation | +6 |
| Delivered measurable business impact | +10 |

**Deductions:** 

| Issue | Points |
|---|---:|
| Avoidable escalation | -5 |
| Repeated handoff without ownership | -3 |

### Formula

$$
P_3 = 5P_s + 4I_p + 8I_m + 5O_w + 6R_p + 10B_i - 5E_a - 3H_o
$$


In [14]:
def score_ownership_impact(
    card,
    problems_solved=0,
    improvements_proposed=0,
    improvements_implemented=0,
    extra_ownership=0,
    risks_prevented=0,
    business_impacts=0,
    avoidable_escalations=0,
    ownership_drop=0,
):
    cat = "Ownership & Impact"
    card.add_points(cat, 5 * problems_solved)
    card.add_points(cat, 4 * improvements_proposed)
    card.add_points(cat, 8 * improvements_implemented)
    card.add_points(cat, 5 * extra_ownership)
    card.add_points(cat, 6 * risks_prevented)
    card.add_points(cat, 10 * business_impacts)
    card.deduct_points(cat, 5 * avoidable_escalations)
    card.deduct_points(cat, 3 * ownership_drop)

## Universal Pillar 4 — Learning & Growth

Measures skill growth and adaptability. **No deductions** by default — we don't
want to penalize people for the act of learning.

### Point rules

| Action | Points |
|---|---:|
| Completed a relevant course / certification | +6 |
| Learned and applied a new tool / skill | +5 |
| Cross-trained into another function | +7 |
| Successfully applied previous feedback | +4 |
| Led a learning session / internal demo | +5 |

### Formula

$$
P_4 = 6C_r + 5S_k + 7X_t + 4F_p + 5L_s
$$


In [15]:
def score_learning_growth(
    card,
    certifications=0,
    new_skills_applied=0,
    cross_training=0,
    feedback_applied=0,
    learning_sessions_led=0,
):
    cat = "Learning & Growth"
    card.add_points(cat, 6 * certifications)
    card.add_points(cat, 5 * new_skills_applied)
    card.add_points(cat, 7 * cross_training)
    card.add_points(cat, 4 * feedback_applied)
    card.add_points(cat, 5 * learning_sessions_led)


## Universal Pillar 5 — Reliability & Work Discipline

Measures consistency, dependability, stakeholder trust, and *sustainable* working.

### Point rules

| Action | Points |
|---|---:|
| Period with strong attendance discipline | +5 |
| Period with no unplanned absence issue | +4 |
| Reliable support during crunch / critical period | +5 |
| Maintained healthy work habits / workload discipline | +3 |
| Positive stakeholder / client appreciation | +4 |

**Deductions:**

| Issue | Points |
|---|---:|
| Unplanned absence affecting continuity | -3 |
| Repeated lateness | -2 |
| Burnout-risk pattern from poor self-management | -3 |

### Formula

$$
P_5 = 5A_d + 4U_a + 5C_p + 3W_h + 4S_a - 3A_b - 2L_t - 3B_r
$$


In [16]:
def score_reliability(
    card,
    good_attendance_periods=0,
    no_unplanned_absence_periods=0,
    crunch_support=0,
    healthy_work_pattern=0,
    stakeholder_appreciation=0,
    absence_issues=0,
    lateness_issues=0,
    burnout_pattern=0,
):
    cat = "Reliability & Work Discipline"
    card.add_points(cat, 5 * good_attendance_periods)
    card.add_points(cat, 4 * no_unplanned_absence_periods)
    card.add_points(cat, 5 * crunch_support)
    card.add_points(cat, 3 * healthy_work_pattern)
    card.add_points(cat, 4 * stakeholder_appreciation)
    card.deduct_points(cat, 3 * absence_issues)
    card.deduct_points(cat, 2 * lateness_issues)
    card.deduct_points(cat, 3 * burnout_pattern)


## Role-Specific Pillars (6, 7, 8)

Each role gets **three additive outcome buckets** mapped to the generic names:
- **Role Output** → core measurable results
- **Role Process Excellence** → doing the work the right way
- **Role Strategic Impact** → long-term / strategic value

The cells below define one scoring function per role. Each takes a scorecard and the
counts of actions achieved, then prices them according to the role's tables.


In [18]:
def score_sales_role(
    card,
    # Output
    qualified_leads=0, opportunities=0, deals_closed=0,
    new_logos=0, quota_milestones=0, upsells=0,
    # Process
    crm_batches=0, discoveries=0, quality_proposals=0, timely_followups=0,
    # Strategic
    exec_contacts=0, referrals=0, expansion_plans=0, strategic_engagements=0,
):
    card.add_points("Role Output", 2 * qualified_leads)
    card.add_points("Role Output", 4 * opportunities)
    card.add_points("Role Output", 12 * deals_closed)
    card.add_points("Role Output", 15 * new_logos)
    card.add_points("Role Output", 20 * quota_milestones)
    card.add_points("Role Output", 8 * upsells)

    card.add_points("Role Process Excellence", 3 * crm_batches)
    card.add_points("Role Process Excellence", 4 * discoveries)
    card.add_points("Role Process Excellence", 5 * quality_proposals)
    card.add_points("Role Process Excellence", 3 * timely_followups)

    card.add_points("Role Strategic Impact", 4 * exec_contacts)
    card.add_points("Role Strategic Impact", 6 * referrals)
    card.add_points("Role Strategic Impact", 5 * expansion_plans)
    card.add_points("Role Strategic Impact", 6 * strategic_engagements)


In [19]:
def score_projects_role(
    card,
    # Output: Delivery Excellence
    milestones_on_time=0, deliverables_in_scope=0,
    risks_mitigated_early=0, successful_releases=0,
    # Process: Governance & Reporting
    status_reports=0, project_docs=0, governance_meetings=0, escalations_in_sla=0,
    # Strategic: Stakeholder & Team Management
    stakeholder_comms=0, dependencies_managed=0,
    knowledge_transfers=0, positive_appreciation=0,
):
    card.add_points("Role Output", 8 * milestones_on_time)
    card.add_points("Role Output", 6 * deliverables_in_scope)
    card.add_points("Role Output", 5 * risks_mitigated_early)
    card.add_points("Role Output", 8 * successful_releases)

    card.add_points("Role Process Excellence", 4 * status_reports)
    card.add_points("Role Process Excellence", 3 * project_docs)
    card.add_points("Role Process Excellence", 4 * governance_meetings)
    card.add_points("Role Process Excellence", 4 * escalations_in_sla)

    card.add_points("Role Strategic Impact", 3 * stakeholder_comms)
    card.add_points("Role Strategic Impact", 5 * dependencies_managed)
    card.add_points("Role Strategic Impact", 4 * knowledge_transfers)
    card.add_points("Role Strategic Impact", 4 * positive_appreciation)


In [20]:
def score_hr_role(
    card,
    # Output: Hiring & Workforce Planning
    positions_filled=0, offers_accepted=0,
    strong_hire_quality=0, internal_moves=0,
    # Process: HR Operations & Compliance
    cases_in_sla=0, training_initiatives=0,
    payroll_cycles_accurate=0, audit_successes=0,
    # Strategic: People Experience & Strategic HR
    coaching_sessions=0, dei_initiatives=0,
    culture_initiatives=0, succession_reviews=0,
):
    card.add_points("Role Output", 10 * positions_filled)
    card.add_points("Role Output", 6 * offers_accepted)
    card.add_points("Role Output", 8 * strong_hire_quality)
    card.add_points("Role Output", 7 * internal_moves)

    card.add_points("Role Process Excellence", 4 * cases_in_sla)
    card.add_points("Role Process Excellence", 4 * training_initiatives)
    card.add_points("Role Process Excellence", 6 * payroll_cycles_accurate)
    card.add_points("Role Process Excellence", 6 * audit_successes)

    card.add_points("Role Strategic Impact", 4 * coaching_sessions)
    card.add_points("Role Strategic Impact", 6 * dei_initiatives)
    card.add_points("Role Strategic Impact", 5 * culture_initiatives)
    card.add_points("Role Strategic Impact", 5 * succession_reviews)


In [21]:
def score_accounts_role(
    card,
    # Output: Financial Accuracy & Reporting
    on_time_close_cycles=0, reconciliations=0,
    variances_explained=0, reports_on_time=0,
    # Process: Controls & Compliance
    control_checks_passed=0, compliance_milestones=0,
    control_cycles_correct=0, risks_flagged=0,
    # Strategic: Stakeholder Service & Improvement
    queries_in_sla=0, process_improvements=0,
    automations=0, support_sessions=0,
):
    card.add_points("Role Output", 10 * on_time_close_cycles)
    card.add_points("Role Output", 4 * reconciliations)
    card.add_points("Role Output", 4 * variances_explained)
    card.add_points("Role Output", 5 * reports_on_time)

    card.add_points("Role Process Excellence", 5 * control_checks_passed)
    card.add_points("Role Process Excellence", 5 * compliance_milestones)
    card.add_points("Role Process Excellence", 4 * control_cycles_correct)
    card.add_points("Role Process Excellence", 6 * risks_flagged)

    card.add_points("Role Strategic Impact", 3 * queries_in_sla)
    card.add_points("Role Strategic Impact", 8 * process_improvements)
    card.add_points("Role Strategic Impact", 10 * automations)
    card.add_points("Role Strategic Impact", 4 * support_sessions)


In [22]:
def score_team_manager_role(
    card,
    # Output: Team Outcomes & Development
    reports_improved=0, promotions_enabled=0, effective_1on1s=0,
    career_plans=0, high_performers_retained=0,
    # Process: Operational Management
    cross_functional_delivered=0, team_sla_targets=0,
    budget_within_plan=0, incidents_in_sla=0,
    # Strategic: Culture & Strategic Leadership
    recognition_given=0, conflicts_resolved=0,
    adoption_led=0, strategic_contributions=0,
):
    card.add_points("Role Output", 6 * reports_improved)
    card.add_points("Role Output", 10 * promotions_enabled)
    card.add_points("Role Output", 3 * effective_1on1s)
    card.add_points("Role Output", 4 * career_plans)
    card.add_points("Role Output", 6 * high_performers_retained)

    card.add_points("Role Process Excellence", 8 * cross_functional_delivered)
    card.add_points("Role Process Excellence", 8 * team_sla_targets)
    card.add_points("Role Process Excellence", 6 * budget_within_plan)
    card.add_points("Role Process Excellence", 4 * incidents_in_sla)

    card.add_points("Role Strategic Impact", 2 * recognition_given)
    card.add_points("Role Strategic Impact", 5 * conflicts_resolved)
    card.add_points("Role Strategic Impact", 6 * adoption_led)
    card.add_points("Role Strategic Impact", 8 * strategic_contributions)


In [23]:
def score_vendor_manager_role(
    card,
    # Output: Vendor Performance & Value
    vendor_sla_targets=0, favorable_renewals=0,
    cost_savings=0, backup_coverage=0,
    # Process: Procurement Process & Compliance
    spend_under_contract=0, rfp_compliant_events=0,
    po_compliance=0, risk_assessments=0,
    # Strategic: Relationship & Strategic Sourcing
    vendor_reviews=0, escalations_resolved=0,
    category_strategies=0, sourcing_initiatives=0,
):
    card.add_points("Role Output", 6 * vendor_sla_targets)
    card.add_points("Role Output", 8 * favorable_renewals)
    card.add_points("Role Output", 10 * cost_savings)
    card.add_points("Role Output", 6 * backup_coverage)

    card.add_points("Role Process Excellence", 6 * spend_under_contract)
    card.add_points("Role Process Excellence", 5 * rfp_compliant_events)
    card.add_points("Role Process Excellence", 4 * po_compliance)
    card.add_points("Role Process Excellence", 4 * risk_assessments)

    card.add_points("Role Strategic Impact", 5 * vendor_reviews)
    card.add_points("Role Strategic Impact", 5 * escalations_resolved)
    card.add_points("Role Strategic Impact", 7 * category_strategies)
    card.add_points("Role Strategic Impact", 6 * sourcing_initiatives)


In [25]:
ROLE_SCORERS = {
    "sales": score_sales_role,
    "projects": score_projects_role,
    "hr": score_hr_role,
    "accounts": score_accounts_role,
    "team_manager": score_team_manager_role,
    "vendor_manager": score_vendor_manager_role,
}


def score_role(card, **kwargs):
    scorer = ROLE_SCORERS.get(card.role)
    if scorer is None:
        raise ValueError(
            f"No scorer registered for role '{card.role}'. "
            f"Available: {list(ROLE_SCORERS)}"
        )
    scorer(card, **kwargs)


## Recognition / Milestone Bonus Layer

To make growth motivating, large achievements grant bonus points on top of pillars.

| Milestone | Bonus |
|---|---:|
| Employee of the month | +15 |
| Client appreciation (key account) | +10 |
| Internal recognition award | +8 |
| Stretch assignment completed | +12 |
| Certified in high-value skill | +10 |
| Saved major risk / cost issue | +15 |

$$
R = \sum_k r_k
$$


In [26]:
BONUS_CATALOG = {
    "employee_of_the_month": 15,
    "client_appreciation": 10,
    "internal_recognition_award": 8,
    "stretch_assignment": 12,
    "high_value_certification": 10,
    "major_risk_cost_save": 15,
}


def add_bonus(card, milestone_key=None, points=None, label="Recognition Bonus"):
    if milestone_key is not None:
        pts = BONUS_CATALOG[milestone_key]
        add_label = milestone_key.replace("_", " ").title()
    else:
        pts = points or 0
        add_label = label
    card.add_points(f"Bonus: {add_label}", pts)


## Guardrails (anti-gaming)

Additive models can be exploited by chasing easy points. These controls keep the score honest.

- **Rule A — Cap repeatable low-value actions** per period (e.g. routine tasks ≤ 20,
  peer-help ≤ 10, CRM batches ≤ 8).
- **Rule B — Require evidence** for every scored action (ticket, approval, CRM record,
  audit log, project artifact).
- **Rule C — No double counting.** One action scores in one category unless designed otherwise.
- **Rule D — Major negatives only.** Don't over-punish small mistakes.

The cell below enforces **Rule A** by clamping selected inputs before scoring.


In [27]:
REPEATABLE_CAPS = {
    "tasks_completed": 20,
    "teammate_support": 10,
    "crm_batches": 8,
    "timely_followups": 15,
}


def apply_caps(**actions):
    capped = dict(actions)
    for key, limit in REPEATABLE_CAPS.items():
        if key in capped and capped[key] > limit:
            print(f"[Guardrail] '{key}' capped: {capped[key]} -> {limit}")
            capped[key] = limit
    return capped


## Interpretation Bands

Since the score is no longer out of 100, we read it with **bands** rather than percentages.
Calibrate these thresholds against real data after 2–3 cycles.

### Monthly bands
| Monthly points | Meaning |
|---|---|
| 0–40 | Low contribution month |
| 41–70 | Solid month |
| 71–100 | Strong month |
| 101–140 | Exceptional month |
| 141+ | Outstanding / rare month |

### Quarterly cumulative bands
| Quarterly cumulative | Meaning |
|---|---|
| 0–150 | Developing |
| 151–260 | Stable contributor |
| 261–380 | Strong contributor |
| 381–500 | High performer |
| 500+ | Exceptional performer |


In [32]:
def monthly_band(score):
    if score <= 40:   return "Low contribution month"
    if score <= 70:   return "Solid month"
    if score <= 100:  return "Strong month"
    if score <= 140:  return "Exceptional month"
    return "Outstanding / rare month"

In [29]:
card = EmployeeScoreCard(name="Asha R.", role="sales", cumulative_score=320)

score_delivery_quality(
    card,
    tasks_completed=12, high_priority_on_time=4, major_deliverables_on_time=2,
    no_rework=3, first_pass_accept=4, complex_work=1, early_completion=1,
)

score_collaboration_culture(
    card,
    teammate_support=3, knowledge_shares=1, positive_peer_feedback=2,
    blockers_resolved=1, culture_participation=1, mentoring=1,
)

score_ownership_impact(
    card,
    problems_solved=2, improvements_proposed=1, improvements_implemented=1,
    extra_ownership=2, risks_prevented=1, business_impacts=1,
)

score_learning_growth(card, certifications=1, new_skills_applied=1, feedback_applied=1)

score_reliability(
    card,
    good_attendance_periods=1, no_unplanned_absence_periods=1,
    crunch_support=1, healthy_work_pattern=1, stakeholder_appreciation=1,
)

sales_actions = apply_caps(
    qualified_leads=10, opportunities=4, deals_closed=2, new_logos=1,
    quota_milestones=1, upsells=1, crm_batches=4, discoveries=3,
    quality_proposals=2, timely_followups=5, exec_contacts=2,
    referrals=1, expansion_plans=1, strategic_engagements=2,
)
score_role(card, **sales_actions)

add_bonus(card, milestone_key="client_appreciation")

result = card.close_month(label="2026-06")
result


{'label': '2026-06',
 'name': 'Asha R.',
 'role': 'sales',
 'monthly_score': 396,
 'cumulative_score': 716,
 'breakdown': {'Delivery & Quality': 88,
  'Collaboration & Culture': 31,
  'Ownership & Impact': 48,
  'Learning & Growth': 15,
  'Reliability & Work Discipline': 21,
  'Role Output': 103,
  'Role Process Excellence': 49,
  'Role Strategic Impact': 31,
  'Bonus: Client Appreciation': 10}}

In [36]:
def print_report(snapshot):
    print(f"Employee : {snapshot['name']}  ({snapshot['role']})")
    print(f"Period   : {snapshot['label']}")
    print("-" * 44)
    for category, pts in snapshot["breakdown"].items():
        print(f"  {category:<32}{pts:>6}")
    print("-" * 44)
    m = snapshot["monthly_score"]
    c = snapshot["cumulative_score"]
    print(f"  {'MONTHLY SCORE':<32}{m:>6}   -> {monthly_band(m)}")
    print(f"  {'CUMULATIVE SCORE':<32}{c:>6}")

print_report(result)

Employee : Asha R.  (sales)
Period   : 2026-06
--------------------------------------------
  Delivery & Quality                  88
  Collaboration & Culture             31
  Ownership & Impact                  48
  Learning & Growth                   15
  Reliability & Work Discipline       21
  Role Output                        103
  Role Process Excellence             49
  Role Strategic Impact               31
  Bonus: Client Appreciation          10
--------------------------------------------
  MONTHLY SCORE                      396   -> Outstanding / rare month
  CUMULATIVE SCORE                   716


In [31]:
import pandas as pd

def history_to_dataframe(card):
    rows = []
    for snap in card.history:
        row = {
            "period": snap["label"],
            "name": snap["name"],
            "role": snap["role"],
            "monthly_score": snap["monthly_score"],
            "cumulative_score": snap["cumulative_score"],
        }
        row.update(snap["breakdown"])
        rows.append(row)
    return pd.DataFrame(rows).fillna(0)

history_to_dataframe(card)


,period,name,role,monthly_score,cumulative_score,Delivery & Quality,Collaboration & Culture,Ownership & Impact,Learning & Growth,Reliability & Work Discipline,Role Output,Role Process Excellence,Role Strategic Impact,Bonus: Client Appreciation
0,2026-06,Asha R.,sales,396,716,88,31,48,15,21,103,49,31,10
